# IOAI — 2024 Mock Competition News Text Classification — ⭐모범답안 (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/jaredliw/ioai-tsp-2025"
CLONE = "ioai-tsp-2025"
SUBDIR = "noai-china-2024/news-text-classification"
WORKDIR = "noai-china-2024/news-text-classification"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

# News Text Classification

> **NOAI China 2024 — Question 3** (APOAI 2025 Mock, Q3).

Classify each news article (`text`) into its `category` with **Word Embedding + LSTM**.
Score = **macro-averaged F1** over all categories (training+inference must fit in ~10 min on CPU).

The official labelled test set is hidden, so we hold out 20% (`index % 5 == 0`) of
`train_news.csv` as validation and report macro-F1 on it. We write `submission_val.csv`
(`idx,category`); the **Submit** button re-scores it against the held-out labels.

In [ ]:
import os, re, random
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

## Load data + held-out validation split (`idx % 5 == 0`)

In [ ]:
df = pd.read_csv('train_news.csv').reset_index(drop=True)
df['idx'] = np.arange(len(df))
is_val = (df['idx'] % 5 == 0)
df_tr, df_va = df[~is_val].copy(), df[is_val].copy()
df['category'].value_counts()

In [ ]:
# Simple, dependency-free tokenizer (no nltk/punkt needed)
def tokenize(text):
    return re.findall(r"[a-z0-9']+", str(text).lower())

MAX_LEN, VOCAB_SIZE = 400, 20000

def build_vocab(texts):
    cnt = Counter(t for txt in texts for t in tokenize(txt))
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for w, _ in cnt.most_common(VOCAB_SIZE - 2):
        vocab[w] = len(vocab)
    return vocab

def encode(texts, vocab):
    out = []
    for txt in texts:
        ids = [vocab.get(t, 1) for t in tokenize(txt)][:MAX_LEN]
        ids += [0] * (MAX_LEN - len(ids))
        out.append(ids)
    return torch.tensor(out, dtype=torch.long)

vocab = build_vocab(df_tr['text'])          # vocab from TRAIN only (no leakage)
le = LabelEncoder().fit(df_tr['category'])
len(vocab), list(le.classes_)

In [ ]:
class TextDS(Dataset):
    def __init__(self, X, y=None):
        self.X = X
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return (self.X[i], self.y[i]) if self.y is not None else self.X[i]

X_tr = encode(df_tr['text'], vocab); y_tr = le.transform(df_tr['category'])
X_va = encode(df_va['text'], vocab)
dl_tr = DataLoader(TextDS(X_tr, y_tr), batch_size=32, shuffle=True)

## Embedding + BiLSTM classifier

In [ ]:
class MyModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2,
                            batch_first=True, bidirectional=True)
        self.fc = nn.Linear(2 * hidden_dim, num_classes)
    def forward(self, x):
        emb = self.embedding(x)
        _, (h, _) = self.lstm(emb)
        last = torch.cat((h[-2], h[-1]), dim=1)
        return self.fc(last)

model = MyModel(len(vocab), 512, 64, len(le.classes_)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)

@torch.no_grad()
def predict(model, X):
    model.eval()
    dl = DataLoader(TextDS(X), batch_size=64)
    out = [model(xb.to(device)).argmax(1).cpu() for xb in dl]
    return torch.cat(out).numpy()

for epoch in range(50):
    model.train()
    for xb, yb in dl_tr:
        xb, yb = xb.to(device), yb.to(device)
        loss = criterion(model(xb), yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    va_pred = le.inverse_transform(predict(model, X_va))
    f1 = f1_score(df_va['category'], va_pred, average='macro')
    print(f'epoch {epoch+1:2d}  loss {loss.item():.4f}  val_macroF1 {f1:.4f}')

In [ ]:
va_pred = le.inverse_transform(predict(model, X_va))
print(f'Held-out validation macro-F1: {f1_score(df_va["category"], va_pred, average="macro"):.4f}')

## Save submission (`submission_val.csv` = `idx,category`)

In [ ]:
sub = pd.DataFrame({'idx': df_va['idx'].to_numpy(), 'category': va_pred})
sub.to_csv('submission_val.csv', index=False)
print('wrote submission_val.csv', sub.shape)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission_val.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)